<a href="https://colab.research.google.com/github/ArjunSingh11994/LLM_RAG/blob/main/LLM_RAG_FINE_TUNE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain transformers sentence-transformers faiss-cpu pypdf streamlit ragas torch pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
loader = PyPDFLoader('/content/yolo.pdf')
document = loader.load()
print(document[0].page_content)

You Only Look Once:
Uniﬁed, Real-Time Object Detection
Joseph Redmon∗, Santosh Divvala∗†, Ross Girshick¶, Ali Farhadi∗†
University of Washington∗, Allen Institute for AI†, Facebook AI Research¶
http://pjreddie.com/yolo/
Abstract
We present YOLO, a new approach to object detection.
Prior work on object detection repurposes classiﬁers to per-
form detection. Instead, we frame object detection as a re-
gression problem to spatially separated bounding boxes and
associated class probabilities. A single neural network pre-
dicts bounding boxes and class probabilities directly from
full images in one evaluation. Since the whole detection
pipeline is a single network, it can be optimized end-to-end
directly on detection performance.
Our uniﬁed architecture is extremely fast. Our base
YOLO model processes images in real-time at 45 frames
per second. A smaller version of the network, Fast YOLO,
processes an astounding 155 frames per second while
still achieving double the mAP of other real-time 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=20)
chunks = text_splitter.split_documents(document)
print(len(chunks))

191


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from langchain_community.vectorstores import FAISS


In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
query = "WHat is yolo?"
docs = retriever.invoke(query)
print(len(docs[0].page_content))

171


In [ ]:
from transformers import pipeline

In [ ]:
llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    tokenizer="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=0,
    max_length=256,

    max_new_tokens=100,
    temperature=0.3,
    do_sample=True,
    top_p=0.95,
    repetition_penalty=1.1
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
#RAG Pipeline

In [ ]:
prompt_template = """
Based on the following context, please answer the question. If the answer cannot be found in the context, respond with "I could not find relevant information."

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
def generate_answer(query):
  docs = retriever.invoke(query)
  context = "\n".join([doc.page_content for doc in docs])
  prompt = prompt_template.format(context=context, question=query)
  answer = llm(prompt)
  return answer[0]["generated_text"]

In [ ]:
chat_history = []
query = "WHat is yolo?"
answer = generate_answer(query)
print(answer)

Both `max_new_tokens` (=100) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Based on the following context, please answer the question. If the answer cannot be found in the context, respond with "I could not find relevant information."

Context:
YOLO is a fast, accurate object detector, making it ideal
for computer vision applications. We connect YOLO to a
webcam and verify that it maintains real-time performance,
same between YOLO and Fast YOLO.
making predictions. Unlike sliding window and region
proposal-based techniques, YOLO sees the entire image
during training and test time so it implicitly encodes contex-
tual information about classes as well as their appearance.

Question:
WHat is yolo?

Answer:
YOLO (You Only Look Once) is a fast, accurate object detection algorithm developed by researchers at UC Berkeley. It uses a deep neural network to predict the location of objects in an image based on their shape, size, and color. The algorithm is trained using a large dataset of labeled images and is capable of detecting a wide range of objects, including ca

In [ ]:
#latency
import time
query = "WHat is yolo?"
answer = generate_answer(query)
start = time.time()
answer = generate_answer(query)
end = time.time()
print("Latency : ",end-start, "Seconds")


Both `max_new_tokens` (=100) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Latency :  4.100660562515259 Seconds


In [ ]:
#evaluation model
# answer evaluation

In [ ]:
test_questions = [

    "What is YOLO?",
    "Why is YOLO fast?",
    "What is object detection?",
    "How does YOLO work?",
    "What are the advantages of YOLO?"

]
for q in test_questions:

    print("\nQUESTION:", q)

    answer = generate_answer(q)

    print("ANSWER:", answer)

    print("-"*50)

Both `max_new_tokens` (=100) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What is YOLO?


Both `max_new_tokens` (=100) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: 
Based on the following context, please answer the question. If the answer cannot be found in the context, respond with "I could not find relevant information."

Context:
YOLO is a fast, accurate object detector, making it ideal
for computer vision applications. We connect YOLO to a
webcam and verify that it maintains real-time performance,
same between YOLO and Fast YOLO.
making predictions. Unlike sliding window and region
proposal-based techniques, YOLO sees the entire image
during training and test time so it implicitly encodes contex-
tual information about classes as well as their appearance.

Question:
What is YOLO?

Answer:
YOLO (You Only Look Once) is a fast, accurate object detector. It makes use of convolutional neural networks (CNNs) to detect objects in an image or video. The algorithm learns to recognize different types of objects by analyzing their features such as size, shape, color, and texture. YOLO can work with images of any size and supports both real-time 

Both `max_new_tokens` (=100) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: 
Based on the following context, please answer the question. If the answer cannot be found in the context, respond with "I could not find relevant information."

Context:
same between YOLO and Fast YOLO.
time that it is so effective at boosting Fast R-CNN’s per-
formance.
Unfortunately, this combination doesn’t beneﬁt from the
speed of YOLO since we run each model seperately and
then combine the results. However, since YOLO is so fast
ing the performance and speed of fast detectors. Fast YOLO is
the fastest detector on record for P ASCAL VOC detection and is
still twice as accurate as any other real-time detector. YOLO is

Question:
Why is YOLO fast?

Answer:
YOLO is extremely fast because it uses a convolutional neural network (CNN) to detect objects in real-time. The CNN is trained on a large dataset of labeled images, which allows it to quickly identify objects in an image. This makes YOLO ideal for real-time applications like video surveillance or autonomous vehicles. Addit

Both `max_new_tokens` (=100) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: 
Based on the following context, please answer the question. If the answer cannot be found in the context, respond with "I could not find relevant information."

Context:
Object detection is a core problem in computer vision.
Detection pipelines generally start by extracting a set of
robust features from input images (Haar [25], SIFT [23],
HOG [4], convolutional features [6]). Then, classiﬁers
object detection and semantic segmentation. In Computer
Vision–ECCV 2014, pages 299–314. Springer, 2014. 7
[8] D. Erhan, C. Szegedy, A. Toshev, and D. Anguelov. Scalable
object detection using deep neural networks. In Computer
Current detection systems repurpose classiﬁers to per-
form detection. To detect an object, these systems take a
classiﬁer for that object and evaluate it at various locations
and scales in a test image. Systems like deformable parts

Question:
What is object detection?

Answer:
Object detection is the process of identifying and localizing objects in an image or vid

Both `max_new_tokens` (=100) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: 
Based on the following context, please answer the question. If the answer cannot be found in the context, respond with "I could not find relevant information."

Context:
YOLO is a fast, accurate object detector, making it ideal
for computer vision applications. We connect YOLO to a
webcam and verify that it maintains real-time performance,
same between YOLO and Fast YOLO.
making predictions. Unlike sliding window and region
proposal-based techniques, YOLO sees the entire image
during training and test time so it implicitly encodes contex-
tual information about classes as well as their appearance.

Question:
How does YOLO work?

Answer:
YOLO works by detecting objects in an image using a convolutional neural network (CNN). The CNN takes an input image and outputs a set of predicted bounding boxes for each object in the image. These bounding boxes are then used to generate a confidence score for each object, which is used to filter out low-confidence predictions. The final outp

In [ ]:
# RETRIEVAL QUALITY EVALUATION

In [ ]:
query = "What is YOLO?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):

    print(f"\nChunk {i+1}\n")

    print(doc.page_content)

    print("="*80)


Chunk 1

YOLO is a fast, accurate object detector, making it ideal
for computer vision applications. We connect YOLO to a
webcam and verify that it maintains real-time performance,

Chunk 2

same between YOLO and Fast YOLO.

Chunk 3

making predictions. Unlike sliding window and region
proposal-based techniques, YOLO sees the entire image
during training and test time so it implicitly encodes contex-
tual information about classes as well as their appearance.
